# CEDAR on CausalRivers discharge measurements

This notebook is the `causalRivers` copy of the `cedar` experiment.
It reads the same frozen, hash-verified bundles as every other method in this
directory and writes one atomic checkpoint per completed dataset fit.

Source: **causalts==0.27.0 from bloomberg/causal-ts commit 0d1fae061b2445a51d0c8b75fc565d897dd54fb7 (installed Git distribution)**. Comparison methods use a pinned installed
distribution when upstream is packageable; repository-only methods call a
verified checkout. Their estimator code is neither copied nor reimplemented
here. Native arrays are saved before common graph-orientation conversion and visualization. This empirical case study compares the inferred directed graphs qualitatively; recovery metrics are not required.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
from pathlib import Path

EXPECTED_PYTHON = (3, 14)
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook requires CPython 3.14; "
        f"found {platform.python_version()}."
    )

import numpy as np

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "src" / "fastCGC" / "__init__.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the fastC-GC project root.")
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from fastCGC.experiments.notebook_checkpoint import (
    configuration_hash,
    derive_algorithm_seed,
    file_sha256,
    reusable_success,
    write_checkpoint,
    write_npz_checkpoint,
)

In [ ]:
DATASET_KEY = "causalRivers"
METHOD_KEY = "cedar"
MASTER_SEED = 42
ALPHA = 0.01
BETA = 0.001
N_PERMUTATIONS = 1000
BOOTSTRAP_SAMPLES = 200

# Pause/resume boundary: one complete method/dataset fit.
RESUME = True
MAX_NEW_FITS: int | None = None

DATASET_DIR = (
    PROJECT_ROOT / "data" / "shared" / "causal-comparison"
    / DATASET_KEY / "datasets"
)
OUTPUT_DIR = (
    PROJECT_ROOT / "data" / "outputs" / "causal-comparison"
    / DATASET_KEY / METHOD_KEY
)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
NATIVE_OUTPUT_DIR = OUTPUT_DIR / "native"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_CONFIGURATION = {
    "dataset": DATASET_KEY,
    "method": METHOD_KEY,
    "master_seed": MASTER_SEED,
    "alpha": ALPHA,
    "beta": BETA,
    "n_permutations": N_PERMUTATIONS,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "truth_scope": "river-network reference",
    "alpha_cond1": ALPHA,
    "alpha_cond2": ALPHA,
    "lag_method": "partial_dcor",
    "lag_pvalue_method": "t_test",
    "lag_alpha": 0.05,
    "lag_n_permutations": 200,
    "normalize": "minmax",
    "prune": True,
    "include_lag0": False,
    "multi_lag": True,
    "multi_lag_keep": "first",
    "include_autoreg": True,
    "assume_ar1": True,
    **({"native_output_schema": 2} if METHOD_KEY.startswith("fast-") else {}),
}
CONFIG_HASH = configuration_hash(RUN_CONFIGURATION)
print(f"Dataset directory: {DATASET_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
import pandas as pd

from fastCGC.experiments.upstream import (
    CAUSALTS_VERSION,
    load_installed_cedar,
)

run_cedar, ParCorrGPU, UPSTREAM_REVISION = load_installed_cedar()
SOURCE_REVISION = UPSTREAM_REVISION.commit


def run_native(series, fit_order, algorithm_seed, run_dir):
    frame = pd.DataFrame(
        np.asarray(series.T, dtype=float),
        columns=[f"X{index}" for index in range(series.shape[0])],
    )
    ci_test = ParCorrGPU(
        frame.to_numpy(copy=False),
        device="cpu",
        seed=algorithm_seed,
    )
    result = run_cedar(
        df=frame,
        ci_test=ci_test,
        max_lag=fit_order,
        alpha_cond1=ALPHA,
        alpha_cond2=ALPHA,
        lag_method="partial_dcor",
        lag_pvalue_method="t_test",
        lag_alpha=0.05,
        n_permutations=200,
        normalize="minmax",
        filter_children=False,
        prune=True,
        sweep_all_lags=False,
        max_prune_path=4,
        include_lag0=False,
        multi_lag=True,
        multi_lag_keep="first",
        n_jobs=1,
        include_C=False,
        include_autoreg=True,
        assume_ar1=True,
        verbose=False,
    )
    return {
        "cg_tig": np.asarray(result.cg_tig),
        "val_matrix": np.asarray(result.val_matrix),
        "pvalue_matrix": np.asarray(result.pvalue_matrix),
        "lag_importance": np.asarray(result.lag_importance),
        "lag_pvalues": np.asarray(result.lag_pvalues),
        "selected_lags": np.asarray(result.selected_lags),
        "var_names": np.asarray(result.var_names),
        "max_lag": np.asarray(result.max_lag),
        "alpha_cond1": np.asarray(result.alpha_cond1),
        "alpha_cond2": np.asarray(result.alpha_cond2),
        "n_tests": np.asarray(result.n_tests),
        "pruned": np.asarray(result.pruned),
        "causalts_version": np.asarray(CAUSALTS_VERSION),
    }

In [ ]:
bundle_paths = sorted(DATASET_DIR.glob("*.npz"))
if not bundle_paths:
    raise FileNotFoundError(
        f"No frozen {DATASET_KEY} bundles found in {DATASET_DIR}. "
        "Run 00-prepare-data.ipynb first."
    )

new_fits = 0
pause_requested = False
for bundle_path in bundle_paths:
    metadata_path = bundle_path.with_suffix(".json")
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Missing metadata for {bundle_path}.")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    with np.load(bundle_path, allow_pickle=False) as stored:
        series = np.asarray(stored["series"], dtype=float)
        if "lag_truth" in stored.files:
            default_order = int(stored["lag_truth"].shape[0])
        else:
            default_order = 1
    digest = hashlib.sha256(
        np.ascontiguousarray(series, dtype=np.float64).tobytes()
    ).hexdigest()
    recorded_digest = metadata.get("data_digest", metadata.get("digest"))
    if digest != recorded_digest:
        raise ValueError(f"Dataset digest mismatch for {bundle_path}.")
    fit_order = int(metadata.get("fit_order", default_order))

    checkpoint = CHECKPOINT_DIR / f"{bundle_path.stem}.json"
    expected = {
        "method": METHOD_KEY,
        "dataset": DATASET_KEY,
        "dataset_id": bundle_path.stem,
        "dataset_digest": digest,
        "fit_order": fit_order,
        "config_hash": CONFIG_HASH,
        "upstream_commit": SOURCE_REVISION,
    }
    if RESUME and reusable_success(checkpoint, expected=expected) is not None:
        print(f"{bundle_path.stem}: checkpoint reused")
        continue
    if MAX_NEW_FITS is not None and new_fits >= MAX_NEW_FITS:
        pause_requested = True
        break

    algorithm_seed = derive_algorithm_seed(
        "causal-comparison",
        MASTER_SEED,
        DATASET_KEY,
        METHOD_KEY,
        bundle_path.stem,
    )
    run_dir = OUTPUT_DIR / "runs" / bundle_path.stem
    run_dir.mkdir(parents=True, exist_ok=True)
    try:
        arrays = run_native(series, fit_order, algorithm_seed, run_dir)
        native_path = NATIVE_OUTPUT_DIR / f"{bundle_path.stem}.npz"
        native_digest = write_npz_checkpoint(native_path, arrays)
        write_checkpoint(
            checkpoint,
            {
                **expected,
                "status": "ok",
                "algorithm_seed": algorithm_seed,
                "native_output": str(native_path.relative_to(PROJECT_ROOT)),
                "native_output_sha256": native_digest,
                "error": "",
            },
        )
        new_fits += 1
        print(f"{bundle_path.stem}: complete")
    except KeyboardInterrupt:
        pause_requested = True
        print("Interrupted. Completed dataset checkpoints are safe.")
        break
    except Exception as error:
        write_checkpoint(
            checkpoint,
            {
                **expected,
                "status": "failed",
                "algorithm_seed": algorithm_seed,
                "native_output": "",
                "native_output_sha256": "",
                "error": f"{type(error).__name__}: {error}",
            },
        )
        print(f"{bundle_path.stem}: failed: {error}")

print(f"Completed {new_fits} new fits in this session.")
if pause_requested:
    print("Rerun this cell to resume at the next unfinished dataset.")